# Anlu Health — license-audited MedGemma QLoRA pilot

This notebook runs a **candidate experiment only** on a Colab GPU. Public datasets and model weights stay in ephemeral Colab storage; validated dataset bundles, adapters, and metrics are written to new UTC-timestamped folders in private Google Drive. Nothing in Drive is deleted or replaced.

Before running: select a GPU runtime, accept the MedGemma terms on Hugging Face, and add `HF_TOKEN` plus a fine-grained, read-only `GH_TOKEN` to Colab Secrets. The GitHub token only needs Contents: Read for the private `LawWeiTin/anlu-health` repository.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU before continuing.'
print(torch.cuda.get_device_name(0))

In [ ]:
!pip -q install 'transformers>=4.53,<5' 'datasets>=4,<5' 'peft>=0.16,<1' 'trl>=0.19,<1' 'bitsandbytes>=0.46,<1' 'accelerate>=1.9,<2' 'huggingface_hub>=0.33,<1' 'PyYAML>=6,<7' 'defusedxml>=0.7,<1'

In [ ]:
import base64, hashlib, json, subprocess
from datetime import datetime, timezone
from pathlib import Path
from google.colab import drive, userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
gh_token = userdata.get('GH_TOKEN')
assert hf_token, 'Add HF_TOKEN to Colab Secrets and grant this notebook access.'
assert gh_token, 'Add a fine-grained read-only GH_TOKEN to Colab Secrets.'
login(token=hf_token, add_to_git_credential=False)
drive.mount('/content/drive', force_remount=False)

REPO_DIR = Path('/content/anlu-health')
if REPO_DIR.exists():
    raise RuntimeError('Use a fresh Colab runtime; refusing to replace an existing checkout.')
header = base64.b64encode(f'x-access-token:{gh_token}'.encode()).decode()
clone = subprocess.run(
    ['git', '-c', f'http.extraheader=AUTHORIZATION: basic {header}', 'clone', '--depth', '1',
     'https://github.com/LawWeiTin/anlu-health.git', str(REPO_DIR)],
    capture_output=True, text=True,
)
del gh_token, header
if clone.returncode:
    raise RuntimeError('Private repository clone failed. Check the GH_TOKEN repository scope.')
REPO_COMMIT = subprocess.run(
    ['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True
).stdout.strip()
print('Repository commit:', REPO_COMMIT)

In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive/Anlu Health')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
DATA_DIR = DRIVE_ROOT / 'Datasets' / RUN_ID
OUTPUT_DIR = DRIVE_ROOT / 'Model Adapters' / RUN_ID
EVAL_DIR = DRIVE_ROOT / 'Evaluation Reports' / RUN_ID
for path in (DATA_DIR, OUTPUT_DIR, EVAL_DIR):
    if path.exists():
        raise RuntimeError(f'Refusing to replace existing Drive content: {path}')

prepare = subprocess.run(
    [
        'python', str(REPO_DIR / 'training/prepare_open_datasets.py'),
        '--manifest', str(REPO_DIR / 'training/open_datasets.yaml'),
        '--behavior-data', str(REPO_DIR / 'training/data/sample_sft.jsonl'),
        '--output-dir', str(DATA_DIR),
        '--work-dir', f'/content/anlu-open-data-{RUN_ID}',
    ],
    cwd=REPO_DIR, capture_output=True, text=True,
)
print(prepare.stdout)
if prepare.returncode:
    raise RuntimeError('Dataset preparation failed: ' + prepare.stderr[-2000:])
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
EVAL_DIR.mkdir(parents=True, exist_ok=False)
print('Run ID:', RUN_ID)
print('Dataset bundle:', DATA_DIR)

In [ ]:
from datasets import load_dataset
dataset = load_dataset(
    'json',
    data_files={
        'train': str(DATA_DIR / 'train.jsonl'),
        'validation': str(DATA_DIR / 'validation.jsonl'),
    },
)
bundle_manifest = json.loads((DATA_DIR / 'dataset_manifest.json').read_text())
assert bundle_manifest['promotion_allowed'] is False
print(dataset)
print(json.dumps(bundle_manifest['audit'], indent=2))

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training

BASE_MODEL = 'google/medgemma-1.5-4b-it'
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
quant = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)
processor = AutoProcessor.from_pretrained(BASE_MODEL, token=hf_token)
model = AutoModelForImageTextToText.from_pretrained(
    BASE_MODEL, token=hf_token, quantization_config=quant, device_map='auto', torch_dtype=compute_dtype
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.config.use_cache = False
lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)

In [ ]:
from trl import SFTConfig, SFTTrainer

def format_example(example):
    return processor.apply_chat_template(
        example['messages'], tokenize=False, add_generation_prompt=False
    )

args = SFTConfig(
    output_dir=str(OUTPUT_DIR / 'checkpoints'),
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    warmup_ratio=0.05,
    lr_scheduler_type='cosine',
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    logging_steps=10,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    max_length=1024,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    optim='paged_adamw_8bit',
    seed=42,
    report_to='none',
)
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    peft_config=lora,
    processing_class=processor,
    formatting_func=format_example,
)

In [ ]:
result = trainer.train()
metrics = trainer.evaluate()
ADAPTER_DIR = OUTPUT_DIR / 'adapter-candidate'
trainer.model.save_pretrained(ADAPTER_DIR)
processor.save_pretrained(ADAPTER_DIR)

def tree_sha256(root):
    digest = hashlib.sha256()
    for path in sorted(item for item in root.rglob('*') if item.is_file()):
        digest.update(path.relative_to(root).as_posix().encode())
        digest.update(path.read_bytes())
    return digest.hexdigest()

dataset_manifest_sha = hashlib.sha256((DATA_DIR / 'dataset_manifest.json').read_bytes()).hexdigest()
candidate = {
    'status': 'candidate_unreviewed',
    'promotion_allowed': False,
    'base_model': BASE_MODEL,
    'repository_commit': REPO_COMMIT,
    'dataset_manifest_sha256': dataset_manifest_sha,
    'adapter_sha256': tree_sha256(ADAPTER_DIR),
    'seed': 42,
    'epochs': 1,
    'metrics': metrics,
    'required_next_gates': [
        'held-out emergency and harmful-advice evaluation',
        'base-model comparison',
        'citation and RAG evaluation',
        'multilingual and TCM safety review',
        'physician, registered TCM practitioner, privacy/security approvals',
    ],
}
(ADAPTER_DIR / 'candidate_manifest.json').write_text(json.dumps(candidate, indent=2, default=str))
(EVAL_DIR / 'training_metrics.json').write_text(json.dumps(metrics, indent=2, default=str))
print(json.dumps(candidate, indent=2, default=str))

## Stop here: candidate only

Training loss is not a medical-safety result. Do not update the production registry or deployment from this notebook. The adapter must first run against the unchanged base model on held-out emergency, harmful-advice, multilingual, citation, RAG, and human-review gates.